# Topic 6 — Text Analytics

**Level 6 Data Science · Unstructured Data**

*What are customers telling us? Extracting insight from unstructured text.*

---

In **Topic 2** you built a pipeline that turns messy text into clean tokens, and you met **TextBlob** for sentiment. Today we use those tools to answer a single business question:

> ### 🎯 *"What should we fix first?"*

A sentiment score alone cannot answer that. So we use sentiment as a **lens** to find the unhappy customers, then ask three questions of what they wrote:

| Question | Technique | Section |
|---|---|---|
| **How** do unhappy customers describe the problem? | TF-IDF distinctive terms | 4 |
| **Who** and what do they name? | Named Entity Recognition (spaCy) | 5 |
| **Which themes** drive the complaints? | Topic modelling (LDA) | 6 |

Then we combine the three answers into a short **insight brief** (Section 7).

### What you will be able to do

- **Understand** the difference between *predicting a label* and *extracting insight* from text *(Understand)*
- **Apply** TF-IDF, Named Entity Recognition and topic modelling to surface what a corpus is about *(Apply)*
- **Analyse** feedback by combining sentiment segmentation with entity and theme analysis *(Analyse)*
- **Evaluate** the limits and ethics of these methods, and judge when NLP is the wrong tool *(Evaluate)*

### Going further

This notebook deliberately covers **three techniques well** rather than many briefly. The **Topic 6 handbook** includes a *Further techniques* toolkit for your own time and your assessment. It covers keyphrase extraction (YAKE, RAKE, TextRank), collocations, keyword-in-context, VADER, aspect-based sentiment, custom entities, clustering, NMF, choosing the number of topics, embeddings and semantic search, and zero-shot classification.

### Accessibility & support

This notebook is provided in an open `.ipynb` format compatible with screen readers, and every figure has a descriptive title and axis labels. If you need an alternative format, contact your tutor. For any wellbeing or safeguarding concerns, speak to your coach or contact `safeguarding@bpp.com`.

## 0. Setup

Alongside the standard stack (**pandas**, **NumPy**, **scikit-learn**, **matplotlib**, **NLTK**) we need two specialist libraries:

- **TextBlob**: ready-made sentiment scoring (Section 3)
- **spaCy**: Named Entity Recognition (Section 5)

```
pip install textblob spacy
python -m spacy download en_core_web_sm
```

We fix a random seed so your results match this notebook, a habit your assessment rewards.

In [ ]:
# If needed (uncomment):
# !pip install pandas numpy scikit-learn nltk matplotlib textblob spacy
# !python -m spacy download en_core_web_sm

import re
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    if not nltk.download(resource, quiet=True):
        print(f"Could not download NLTK '{resource}'. If you are behind a work proxy, "
              "recent NLTK versions refuse proxied downloads: set the environment "
              "variable NLTK_ALLOW_PROXIED_URLOPEN=1 and restart the kernel.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 90)
print("Setup complete.")

## 1. Recap: the Topic 2 preprocessing pipeline

We reuse the Topic 2 preprocessing function unchanged. One idea matters for today: **how much you clean depends on the task.**

- **TF-IDF** and **topic modelling** count words, so they use the *cleaned* tokens (lowercased, stop words removed, lemmatised).
- **Named Entity Recognition** uses the **raw** text, because capitalisation is how it recognises a name.

In [ ]:
STOP_WORDS = set(stopwords.words("english"))
LEMMATISER = WordNetLemmatizer()

def preprocess_text(text: str) -> list[str]:
    """Topic 2 pipeline: clean a string and return a list of lemmatised tokens."""
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)   # URLs
    text = re.sub(r"\S+@\S+\.\S+", "", text)             # emails
    text = re.sub(r"<.*?>", "", text)                     # HTML
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)           # punctuation / emoji
    text = re.sub(r"\s+", " ", text).strip()
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in STOP_WORDS]
    tokens = [LEMMATISER.lemmatize(w) for w in tokens]
    return tokens

def preprocess_to_string(text: str) -> str:
    """Cleaned tokens joined back into a string (the form scikit-learn expects)."""
    return " ".join(preprocess_text(text))

messy = "Currys in Manchester were GREAT!! Sorted my refund fast :) https://shop.example.com"
print("Raw:    ", messy)
print("Cleaned:", preprocess_to_string(messy))
print("Note how cleaning would strip 'Currys'/'Manchester' capitalisation — which is why NER uses raw text.")

## 2. The dataset: customer reviews

We continue with **customer reviews**. These synthetic reviews mention real-world **entities** (products, retailers, couriers, places), and each centres on one of five **themes**: delivery, product quality, price/value, customer service, and the online experience.

> **Note.** The data is synthetic so the notebook runs anywhere. Each review carries a hidden `theme` label. We do **not** use it in the analysis; we keep it only to check later how well LDA recovers the themes.

In [ ]:
def make_reviews(n: int = 300, seed: int = SEED) -> pd.DataFrame:
    """Generate a synthetic but entity-rich, multi-theme customer-reviews dataset."""
    rng = random.Random(seed)
    products  = ["Sony television", "Samsung phone", "Nike trainers", "Dyson vacuum",
                 "Ikea desk", "Bosch dishwasher", "Apple laptop", "Philips kettle"]
    retailers = ["Currys", "Argos", "Amazon", "John Lewis", "Very", "AO"]
    couriers  = ["DPD", "Royal Mail", "Evri", "Yodel", "DHL"]
    places    = ["Manchester", "London", "Leeds", "Bristol", "Cardiff", "Glasgow"]
    emoji     = ["\U0001F600", "\U0001F620", "\U0001F44D", "\U0001F494"]

    themes = {
        "delivery": {
            "pos": ["Delivery by {courier} was quick and the {product} arrived a day early.",
                    "The {product} turned up on time and {courier} kept me updated throughout.",
                    "Fast dispatch from {retailer} and {courier} delivered to {place} without a hitch."],
            "neg": ["The {product} arrived a week late and {courier} left it out in the rain.",
                    "{courier} lost my parcel and {retailer} took ages to resend the {product}.",
                    "Delivery to {place} was a nightmare, {courier} missed three slots."]},
        "quality": {
            "pos": ["The {product} from {retailer} is beautifully made and feels really sturdy.",
                    "Superb build quality on the {product}, it feels premium and solid.",
                    "The {product} is well made and the materials are genuinely excellent."],
            "neg": ["The {product} broke after two days, really flimsy and poor quality.",
                    "Cheap materials on the {product}, it felt faulty straight out of the box.",
                    "The {product} from {retailer} fell apart quickly, terrible quality."]},
        "price": {
            "pos": ["Brilliant value, the {product} was much cheaper at {retailer} than elsewhere.",
                    "Great price for the {product}, genuinely worth the money.",
                    "The {product} was a real bargain in the {retailer} sale."],
            "neg": ["Overpriced for what it is, the {product} is not worth the money.",
                    "The {product} at {retailer} was far too expensive for the quality.",
                    "Felt ripped off, the {product} costs half as much elsewhere."]},
        "service": {
            "pos": ["The staff at {retailer} in {place} were friendly and really helpful.",
                    "Customer service was excellent and my refund on the {product} was fast.",
                    "Contacted {retailer} about the {product} and support sorted it quickly."],
            "neg": ["Customer service at {retailer} was rude and unhelpful about my {product}.",
                    "Waited hours for support and still no refund on the faulty {product}.",
                    "The staff in {place} were dismissive when the {product} arrived damaged."]},
        "online": {
            "pos": ["Ordering the {product} on the {retailer} app was smooth and easy.",
                    "The {retailer} website made checkout simple and tracking the {product} was clear.",
                    "Loved how easy the {retailer} app made reordering the {product}."],
            "neg": ["The {retailer} website kept crashing at checkout for the {product}.",
                    "Awful app, my account locked and I could not track the {product}.",
                    "The {retailer} online checkout was confusing and lost my {product} order."]},
    }

    rows = []
    for _ in range(n):
        theme = rng.choice(list(themes))
        polarity = rng.choice(["pos", "neg"])
        text = rng.choice(themes[theme][polarity]).format(
            product=rng.choice(products), retailer=rng.choice(retailers),
            courier=rng.choice(couriers), place=rng.choice(places))
        # light, realistic noise
        if rng.random() < 0.12: text += " " + rng.choice(emoji)
        if rng.random() < 0.08: text = f"<b>{text}</b>"
        if rng.random() < 0.06: text += " https://reviews.example.co.uk"
        rows.append({"review": text, "theme": theme})
    return pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)

df = make_reviews()
print(f"Dataset: {len(df)} reviews across {df['theme'].nunique()} hidden themes")
df.head(8)

## 3. Sentiment as a lens

You already have tools that score sentiment, so we will **not** build a classifier. **TextBlob** gives each review a **polarity** from −1 (very negative) to +1 (very positive). We use it to split the corpus into groups, and every later section compares those groups.

> **Prediction vs insight.** A sentiment model *predicts a label* for one review. Insight extraction *describes a whole collection* so that a person can decide what to do. Tools already handle the label well. The insight is the harder, more valuable work, and it is what your assessment rewards.

In [ ]:
from textblob import TextBlob

df["polarity"] = df["review"].apply(lambda t: TextBlob(t).sentiment.polarity)
df["sentiment"] = pd.cut(df["polarity"], bins=[-1.01, -0.05, 0.05, 1.01],
                         labels=["negative", "neutral", "positive"])

print(df["sentiment"].value_counts(), "\n")
for _, r in df.sample(5, random_state=SEED).iterrows():
    print(f"  {r['polarity']:+.2f}  {r['review'][:70]}")

> 🔍 **Look closely at the scores.** TextBlob gives some obvious complaints a score near zero. *"Yodel lost my parcel…"* lands in **neutral**, because none of its words appear in TextBlob's lexicon. A lens is only as good as its lexicon. Keep this in mind when you read the results, and see the handbook's comparison of TextBlob with **VADER**.

## 4. *How* do they describe it? Distinctive terms with TF-IDF

Raw word counts are dominated by words common to **every** review, so the positive and negative lists would look almost identical. **TF-IDF** gives more weight to words that are *distinctive*:

$$w_{x,y} = tf_{x,y} \times \log\left(\frac{N}{df_x}\right)$$

Words that appear in almost every review get pushed down; words concentrated in a few reviews get lifted. Averaging the weights within each sentiment group shows what characterises that group.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

df["clean"] = df["review"].apply(preprocess_to_string)

vec = TfidfVectorizer(min_df=3)
tfidf = vec.fit_transform(df["clean"])
terms = np.array(vec.get_feature_names_out())

def distinctive_terms(mask: pd.Series, k: int = 10) -> list[str]:
    """Mean TF-IDF weight of each term within a subset of reviews; return the top k."""
    mean_weights = np.asarray(tfidf[mask.values].mean(axis=0)).ravel()
    return list(terms[np.argsort(mean_weights)[-k:][::-1]])

top_pos_terms = distinctive_terms(df["sentiment"] == "positive")
top_neg_terms = distinctive_terms(df["sentiment"] == "negative")
print("Distinctive of POSITIVE reviews:", ", ".join(top_pos_terms))
print("Distinctive of NEGATIVE reviews:", ", ".join(top_neg_terms))

> **Knowledge check.** Why is TF-IDF more useful than raw counts for *comparing* the positive and negative groups?
>
> <details><summary>Show answer</summary>
>
> Raw counts are dominated by words common to both groups (product names, "order", "arrived"), so the two lists look almost identical. TF-IDF gives less weight to those shared words and more to the terms *distinctive* to each group, which is exactly the contrast we want.
> </details>

## 5. *Who* do they name? Named Entity Recognition

**Named Entity Recognition (NER)** finds and classifies real-world entities such as organisations (`ORG`), places (`GPE`) and products (`PRODUCT`). Combined with our sentiment lens, it tells us **which brands and couriers attract complaints**.

We run spaCy on the **raw** text, removing only the HTML tags, which are noise and not a signal. Then we report a **complaint rate** for each entity: the share of its mentions that come from negative reviews. A raw count would simply favour whichever brand is mentioned most.

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")   # if this fails: python -m spacy download en_core_web_sm

entity_rows = []
for idx, text in df["review"].items():
    raw = re.sub(r"<.*?>", "", text)                # strip HTML only — keep the casing
    for ent in nlp(raw).ents:
        entity_rows.append({"entity": ent.text, "label": ent.label_,
                            "negative": df.at[idx, "sentiment"] == "negative"})
ent_df = pd.DataFrame(entity_rows)

print(f"Extracted {len(ent_df)} entity mentions. By type:")
print(ent_df["label"].value_counts().head(6).to_string(), "\n")

orgs = (ent_df[ent_df["label"] == "ORG"]
        .groupby("entity")["negative"].agg(mentions="size", complaints="sum"))
orgs["complaint_rate"] = (orgs["complaints"] / orgs["mentions"]).round(2)
entity_table = (orgs[orgs["mentions"] >= 10]
                .sort_values("complaint_rate", ascending=False))
print("Organisations by complaint rate (10+ mentions):")
print(entity_table.to_string())

> ⚠️ **Two cautions.**
> 1. **NER makes mistakes.** Look at the label counts: the small model tags some brands as `PERSON` and misses others entirely. Check a sample by eye before trusting the totals. The handbook shows how spaCy's `EntityRuler` adds your own known brands.
> 2. **NER extracts personal data.** Once you pull `PERSON` or `GPE` entities from real feedback, you are processing information governed by **UK GDPR**. Handle and store it accordingly.

> **Knowledge check.** Should NER be run on cleaned or raw text?
>
> <details><summary>Show answer</summary>
>
> **Raw.** NER treats capitalisation as a signal that a word is a name. Lowercasing and stripping punctuation first would remove exactly the information the model needs.
> </details>

## 6. *Which themes* drive complaints? Topic modelling with LDA

Reading every review does not scale. **Latent Dirichlet Allocation (LDA)** discovers recurring themes **without labels** (it is *unsupervised*). It treats each review as a mixture of topics and each topic as a set of words that tend to appear together.

🔍 **A piece of judgement:** brand and place names are *entities*, not *themes*. If we leave them in, they blur the topics, so we remove them as **domain-specific stop words** for this step.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

# Entity names are not themes. Note the lemmatised forms ("curry", "philip").
ENTITY_STOPWORDS = ["sony", "samsung", "nike", "dyson", "ikea", "bosch", "apple", "philips",
                    "philip", "television", "phone", "trainer", "vacuum", "desk", "dishwasher",
                    "laptop", "kettle", "currys", "curry", "argos", "amazon", "john",
                    "lewis", "ao", "dpd", "royal", "mail", "evri", "yodel", "dhl",
                    "manchester", "london", "leeds", "bristol", "cardiff", "glasgow"]

cvec = CountVectorizer(max_df=0.9, min_df=3, stop_words=ENTITY_STOPWORDS)
counts = cvec.fit_transform(df["clean"])
words = np.array(cvec.get_feature_names_out())

lda = LatentDirichletAllocation(n_components=5, random_state=SEED)
df["topic"] = lda.fit_transform(counts).argmax(axis=1) + 1

topic_top_words = {}
for i, weights in enumerate(lda.components_, start=1):
    topic_top_words[i] = [words[j] for j in weights.argsort()[-7:][::-1]]
    print(f"Topic {i}: {', '.join(topic_top_words[i])}")

LDA does **not** name its topics. *You* read the top words and give each topic a label. The labels below are ours; yours may differ, and that is fine as long as you can justify them from the words.

In [ ]:
# Human labels, read from the top words above. Edit them if you see it differently.
TOPIC_LABELS = {1: "Bargains & speed", 2: "Overpriced", 3: "Service & refunds",
                4: "Delivery & damage", 5: "Online checkout"}

topic_sentiment = pd.crosstab(df["topic"], df["sentiment"])
topic_sentiment["complaint_rate"] = (topic_sentiment["negative"]
                                     / topic_sentiment.sum(axis=1)).round(2)
topic_sentiment.index = [f"{t}. {TOPIC_LABELS[t]}" for t in topic_sentiment.index]
print(topic_sentiment.sort_values("complaint_rate", ascending=False))

fig, ax = plt.subplots(figsize=(8, 3.5))
rates = topic_sentiment["complaint_rate"].sort_values()
ax.barh(rates.index, rates.values, color="#2E6DA4")
ax.set_title("Complaint rate by discovered theme")
ax.set_xlabel("Share of the theme's reviews that are negative")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.tight_layout()
plt.show()

> **Caveat.** Topics are *statistical*, not semantic, and the results change with the number of topics and the preprocessing. As a sanity check, the cell below compares each topic with the hidden themes. The match is only partial: price appears twice and quality has no topic of its own. Treat topic modelling as a lens for exploration, never as ground truth.

In [ ]:
match = df.groupby("topic")["theme"].agg(lambda s: f"{s.value_counts().index[0]} "
                                                   f"({s.value_counts().iloc[0] / len(s):.0%})")
print("Most common hidden theme in each discovered topic:")
print(match.to_string())

## 7. Bringing it together: the insight brief

Each technique on its own gives you a signal. The value comes from **combining** them into an answer to our question: *what should we fix first?*

In [ ]:
worst_topic = topic_sentiment["complaint_rate"].idxmax()
worst_orgs = entity_table.head(3)

print("=" * 62)
print("INSIGHT BRIEF — what should we fix first?")
print("=" * 62)
print(f"Corpus: {len(df)} reviews  |  "
      + ", ".join(f"{k}: {v}" for k, v in df['sentiment'].value_counts().items()))
print(f"\nHOW they describe it:   {', '.join(top_neg_terms[:8])}")
print("\nWHO they name most in complaints (complaint rate):")
for name, row in worst_orgs.iterrows():
    print(f"   {name:<12} {row['complaint_rate']:.0%} of {int(row['mentions'])} mentions")
print(f"\nWHICH theme: {worst_topic} — "
      f"{topic_sentiment.loc[worst_topic, 'complaint_rate']:.0%} of its reviews negative")
print("=" * 62)

> 📌 **Look at what the brief does.** It *names* the problem area, *quantifies* it with a rate rather than a raw count, and *captures the customers' own language*. A manager can act on that; a sentiment score alone would not tell them where to start.

## 8. Limits, ethics and when NLP is the wrong tool

**When a simpler or different approach wins:**

- **Interpretability.** If every decision must be auditable, a transparent rule-based keyword search may beat an opaque model.
- **Volume.** With a few hundred reviews a month, the classical methods here are cheaper and clearer than a transformer.
- **Language coverage.** English-centric tools (spaCy's small model, TextBlob, NLTK stop words) quietly under-perform on dialect, slang and other languages. A more complex model will not fix a language-coverage gap.

**Ethics.** Lexicons and entity models reflect their training data: you saw TextBlob miss *"lost my parcel"*. NER extracts personal data and so engages **UK GDPR**. If preprocessing strips slang or cultural expression, some customers may disappear from your "insight". Ask whose voice your pipeline might be losing.

## 9. Exercises

**Core**

1. **Add context with bigrams.** Re-run Section 4 with `TfidfVectorizer(ngram_range=(1, 2), min_df=3)`. Which two-word phrases (e.g. *"account locked"*) characterise complaints, and what noise appears alongside them?
2. **Couriers only.** Filter `orgs` (not `entity_table`, which drops anything with fewer than 10 mentions) to the five couriers: DPD, Royal Mail, Evri, Yodel, DHL. Which fares worst, and how far would you trust a rate based on so few mentions?
3. **Sharper lens.** Keep only strongly positive or negative reviews (|polarity| > 0.3). Do the distinctive terms change?
4. **Your labels.** Rewrite `TOPIC_LABELS` in your own words and justify each label from the top words.

**Stretch (use the handbook's *Further techniques* section)**

5. **Choose k.** Re-run LDA with 3, 5 and 8 topics. How would you decide the "right" number? (Research topic *coherence*.)
6. **Fix the lens.** Swap TextBlob for VADER and compare how many complaints each finds.
7. **Bring your own data.** Point the workflow at a real text set and write a one-paragraph insight brief.

## 10. Summary & where this connects

**What you did.** You answered *"what should we fix first?"* by segmenting reviews with TextBlob, then asking **how** unhappy customers describe the problem (TF-IDF), **who** they name (NER) and **which themes** drive complaints (LDA). You combined the answers into an insight brief and weighed the limits.

📌 **Link to your summative assessment.** This workflow is a template for **Section A (code scripts)** and the evidence in **Section B**. To reach the higher mark bands:

- **extract insight**, not just scores: connect entities, themes and language to a business question;
- present method choices as a **reasoned trade-off**, and say where a technique is the *wrong* tool;
- keep the workflow **reproducible and well-documented**; and
- reference sources using **Harvard** style.

**Further reading:** the **Topic 6 handbook** (core techniques plus the *Further techniques* toolkit) · spaCy *Linguistic Features* · scikit-learn *LatentDirichletAllocation* · TextBlob documentation.